In [0]:
scope="secretscope"
key="engdados-secret"
storage_account="stoengdados"
application_id = dbutils.secrets.get(scope, "application-id")
directory_id = dbutils.secrets.get(scope, "tenant-id")
service_credential = dbutils.secrets.get(scope=scope,key=key)

spark.conf.set("fs.azure.account.auth.type.%s.dfs.core.windows.net"%(storage_account), "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.%s.dfs.core.windows.net"%(storage_account), "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.%s.dfs.core.windows.net"%(storage_account), application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.%s.dfs.core.windows.net"%(storage_account), service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.%s.dfs.core.windows.net"%(storage_account), "https://login.microsoftonline.com/%s/oauth2/token"%(directory_id))



In [0]:
container_name = "lakehouse"
# Get all the files under the ADLS folder and create a list of file paths
file_list = dbutils.fs.ls("abfss://%s@%s.dfs.core.windows.net/silver"%(container_name,storage_account))

# Read each file and create a DataFrame
for file_path in file_list:
    print(file_path)
    df = spark.read.format("delta").load(path=file_path.path[:-1])
    # You can process the DataFrame or register it as a table here
    # For example, to create a temporary table:
    df.createOrReplaceTempView(file_path.name[:-1])


FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/salesCustomer/', name='salesCustomer/', size=0, modificationTime=1774407001000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/salesCustomerAddress/', name='salesCustomerAddress/', size=0, modificationTime=1774407005000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/salesOrderDetail/', name='salesOrderDetail/', size=0, modificationTime=1774406995000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/salesOrderHeader/', name='salesOrderHeader/', size=0, modificationTime=1774406954000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/salesProduct/', name='salesProduct/', size=0, modificationTime=1774407009000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/salesProductCategory/', name='salesProductCategory/', size=0, modificationTime=1774407013000)


In [0]:
%sql
SHOW VIEWS

namespace,viewName,isTemporary,isMaterialized
,salescustomer,true,false
,salescustomeraddress,true,false
,salesorderdetail,true,false
,salesorderheader,true,false
,salesproduct,true,false
,salesproductcategory,true,false


In [0]:
from pyspark.sql.functions import monotonically_increasing_id
#Create DimCustomer
df_dimCustomer = spark.sql("select sc.* , sca.AddressID, sca.AddressType from salescustomer sc join salescustomeraddress sca on sc.customerid = sca.customerid")
#Add surrogate key as the first column
df_dimCustomer_with_surrogate_key = df_dimCustomer.withColumn("CustomerIDKey", monotonically_increasing_id())\
    .select(
        "CustomerIDKey",  # Select the surrogate key column first
        *[column for column in df_dimCustomer.columns if column != "CustomerIDKey"]  # Select the remaining columns in their original order
    )

In [0]:
from pyspark.sql.functions import monotonically_increasing_id
#Create dimProduct
df_dimProduct = spark.sql("select sp.*, spc.ParentProductCategoryID, spc.Name as ProductCategoryName from salesproduct sp join salesproductcategory spc on sp.ProductCategoryID = spc.ProductCategoryID")
#Add surrogate key as the first column
df_dimProduct_with_surrogate_key = df_dimProduct.withColumn("ProductIDKey", monotonically_increasing_id())\
    .select(
        "ProductIDKey",  # Select the surrogate key column first
        *[column for column in df_dimProduct.columns if column != "ProductIDKey" and column!="spc.Name"]  # Select the remaining columns in their original order
    )

In [0]:
from pyspark.sql.functions import expr
 
# Define the start and end dates for your DimDate table
start_date = "2000-01-01"
end_date = "2024-12-31"
 
# Create a DataFrame with a range of dates
df_dimDate = spark.range(0, (spark.sql("SELECT datediff('{0}', '{1}')".format(end_date, start_date)).collect()[0][0])+1) \
    .selectExpr("CAST(id AS INT) AS id") \
    .selectExpr("date_add('{0}', id) AS Date".format(start_date))
 
# Extract different date components
df_dimDate = df_dimDate \
    .withColumn("Year", expr("year(Date)")) \
    .withColumn("Month", expr("month(Date)")) \
    .withColumn("DayOfMonth", expr("dayofmonth(Date)")) \
    .withColumn("DayOfYear", expr("dayofyear(Date)")) \
    .withColumn("WeekOfYear", expr("weekofyear(Date)")) \
    .withColumn("DayOfWeek", expr("dayofweek(Date)")) \
    .withColumn("Quarter", expr("quarter(Date)"))


#Write dataframes to Gold Layer as External Tables

In [0]:
# Write dimProduct table into ADLS Gold layer as External Tables

path="abfss://%s@%s.dfs.core.windows.net/gold"%(container_name,storage_account)

tableName="dimProduct"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
df_dimProduct_with_surrogate_key.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
#df_dimProduct_with_surrogate_key.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")


In [0]:
# Write dimCustomer table into ADLS Gold layer as External Tables

path="abfss://%s@%s.dfs.core.windows.net/gold"%(container_name,storage_account)

tableName="dimCustomer"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
df_dimCustomer_with_surrogate_key.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
#df_dimCustomer_with_surrogate_key.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")


In [0]:
# Write dimDate table into ADLS Gold layer as External Tables

path="abfss://%s@%s.dfs.core.windows.net/gold"%(container_name,storage_account)

tableName="dimDate"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
# df_dimDate.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_dimDate.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")

In [0]:
#Create factSales
df_factSales = spark.sql("select dp.ProductIDKey, ds.CustomerIDKey, soh.*, sod.OrderQty, sod.ProductID, sod.UnitPrice, sod.UnitPriceDiscount, sod.LineTotal from salesorderheader soh join salesorderdetail sod on soh.SalesOrderID = sod.SalesOrderID LEFT JOIN default.dimProduct dp ON sod.ProductID = dp.ProductID LEFT JOIN default.dimCustomer ds ON soh.CustomerID = ds.CustomerID")

In [0]:
# Write factSales table into ADLS Gold layer as External Tables

path="abfss://%s@%s.dfs.core.windows.net/gold"%(container_name,storage_account)

tableName="factSales"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
#df_factSales.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_factSales.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")
